
# Hilvan, Türkiye M5.3 — FDSN waveform acquisition + Coulomb stress workflow

**Event:** M 5.3 — 15 km SSW of Hilvan, Türkiye  
**USGS event ID:** `us6000tx9u`  
**Origin:** 2026-09-24 07:40:58 UTC  
**Epicenter:** 37.4736°N, 38.8612°E  
**Depth:** 10.0 km  
**Station search radius:** 500 km

This notebook:

1. Downloads the USGS event record and QuakeML.
2. Downloads open FDSN waveform data and StationXML from stations within 500 km.
3. Explicitly includes KOERI's EIDA FDSN endpoint and also tries ObsPy-known FDSN providers.
4. Creates station/channel, distance, azimuth, and waveform audit tables.
5. Converts MiniSEED to SAC with event/station headers.
6. Optionally removes instrument response to ground velocity.
7. Extracts USGS phase picks/arrivals when available.
8. Extracts a USGS moment-tensor/focal-mechanism nodal plane when available.
9. Computes an Okada elastic-half-space Coulomb failure stress (ΔCFS) scenario.
10. Produces separate maps for NP1 and NP2 because the moment tensor alone does not identify the physical rupture plane.

> **Important:** the Coulomb section is a scenario calculation, not an earthquake prediction. Results depend strongly on rupture geometry, receiver-fault orientation, friction, stress drop, depth, and elastic parameters.


### Repository use
This notebook is the public-data acquisition workflow. When run from the repository root it writes
downloaded material under `work/Hilvan_M53_us6000tx9u/`. Raw waveform redistribution is not part
of this release; users should download from the original FDSN/network providers and cite them.


## References

- FDSN web services: https://www.fdsn.org/webservices/
- USGS FDSN event service: https://earthquake.usgs.gov/fdsnws/event/1/
- USGS event page: https://earthquake.usgs.gov/earthquakes/eventpage/us6000tx9u/executive
- KOERI FDSN data center: https://www.fdsn.org/datacenters/detail/KOERI/
- KOERI dataselect: https://eida.koeri.boun.edu.tr/fdsnws/dataselect/1/
- ObsPy FDSN clients and MassDownloader: https://docs.obspy.org/packages/obspy.clients.fdsn.html
- Pyrocko Okada model: https://pyrocko.org/docs/current/library/reference/pyrocko.modelling.okada.html
- Okada, Y. (1992), *Internal deformation due to shear and tensile faults in a half-space*.


In [ ]:

# Install only missing packages.
import sys, subprocess, importlib.util

REQUIRED = {
    "obspy": "obspy>=1.4.2",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "requests": "requests",
    "pyproj": "pyproj",
    "tqdm": "tqdm",
    "pyrocko": "pyrocko>=2026.6.2",
}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]

if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *missing])
else:
    print("All required packages are available.")

if importlib.util.find_spec("cartopy") is None:
    print("Optional for coastlines/borders: pip install cartopy")


In [ ]:

from pathlib import Path
import json, math, traceback
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from tqdm.auto import tqdm

from obspy import UTCDateTime, read, read_inventory
from obspy.core import AttribDict
from obspy.clients.fdsn import Client
from obspy.clients.fdsn.header import URL_MAPPINGS
from obspy.clients.fdsn.mass_downloader import CircularDomain, Restrictions, MassDownloader
from obspy.geodetics import gps2dist_azimuth, kilometers2degrees
from obspy.geodetics.base import locations2degrees

from pyproj import CRS, Transformer

EVENT_ID = "us6000tx9u"
EVENT_TIME = UTCDateTime("2026-09-24T07:40:58")
EVENT_LAT = 37.4736
EVENT_LON = 38.8612
EVENT_DEPTH_KM = 10.0
EVENT_MAG = 5.3

RADIUS_KM = 500.0
PRE_EVENT_SECONDS = 120
POST_EVENT_SECONDS = 1200

ROOT = Path("work") / f"Hilvan_M53_{EVENT_ID}"
DIRS = {
    "event": ROOT / "event_metadata",
    "mseed": ROOT / "waveforms_mseed",
    "stationxml": ROOT / "stationxml",
    "sac_raw": ROOT / "sac_raw",
    "sac_vel": ROOT / "sac_velocity",
    "tables": ROOT / "tables",
    "figures": ROOT / "figures",
    "coulomb": ROOT / "coulomb",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Project:", ROOT.resolve())



## 1. USGS detail product and QuakeML

The USGS detail product may gain additional products during later review. Re-run this section later to pick up a new focal mechanism, phase-data, or revised origin.


In [ ]:

USGS_DETAIL_URL = f"https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/{EVENT_ID}.geojson"

session = requests.Session()
session.headers.update({
    "User-Agent": "Hilvan-M53-FDSN-research-notebook/1.0 (academic seismology)"
})

detail = None
try:
    r = session.get(USGS_DETAIL_URL, timeout=60)
    r.raise_for_status()
    detail = r.json()
    with open(DIRS["event"] / f"{EVENT_ID}_detail.geojson", "w", encoding="utf-8") as f:
        json.dump(detail, f, indent=2)
    p = detail.get("properties", {})
    print("Title:", p.get("title"))
    print("Magnitude:", p.get("mag"), p.get("magType"))
    print("Status:", p.get("status"))
    print("Products:", sorted(p.get("products", {}).keys()))
except Exception as e:
    print("USGS detail download failed:", repr(e))
    print("Continuing with the event constants at the top of the notebook.")


In [ ]:

usgs = Client("USGS", timeout=120)

try:
    try:
        cat_event = usgs.get_events(eventid=EVENT_ID, includearrivals=True)
    except Exception:
        cat_event = usgs.get_events(eventid=EVENT_ID)

    qml_path = DIRS["event"] / f"{EVENT_ID}.xml"
    cat_event.write(str(qml_path), format="QUAKEML")
    print(cat_event)
    print("Saved:", qml_path)
except Exception as e:
    cat_event = None
    print("QuakeML request failed:", repr(e))



## 2. Phase picks / arrivals from QuakeML


In [ ]:

phase_rows = []

if cat_event:
    for ev in cat_event:
        pick_lookup = {str(p.resource_id): p for p in ev.picks}
        for org in ev.origins:
            for arr in org.arrivals:
                pick = pick_lookup.get(str(arr.pick_id))
                wid = getattr(pick, "waveform_id", None) if pick else None
                phase_rows.append({
                    "origin_time": str(org.time) if org.time else None,
                    "phase": arr.phase,
                    "pick_time": str(pick.time) if pick and pick.time else None,
                    "network": getattr(wid, "network_code", None) if wid else None,
                    "station": getattr(wid, "station_code", None) if wid else None,
                    "location": getattr(wid, "location_code", None) if wid else None,
                    "channel": getattr(wid, "channel_code", None) if wid else None,
                    "time_residual_s": arr.time_residual,
                    "azimuth_deg": arr.azimuth,
                    "distance_deg": arr.distance,
                })

phases_df = pd.DataFrame(phase_rows)
if not phases_df.empty:
    phases_df.to_csv(DIRS["tables"] / f"{EVENT_ID}_phase_arrivals.csv", index=False)
    display(phases_df.head(30))
    print("Arrivals:", len(phases_df))
else:
    print("No phase arrivals are present in the current QuakeML product.")



## 3. Regional catalog for foreshocks / aftershocks


In [ ]:

CATALOG_DAYS_BEFORE = 30
CATALOG_DAYS_AFTER = 7
MIN_CATALOG_MAG = 1.5

try:
    cat_regional = usgs.get_events(
        starttime=EVENT_TIME - CATALOG_DAYS_BEFORE * 86400,
        endtime=EVENT_TIME + CATALOG_DAYS_AFTER * 86400,
        latitude=EVENT_LAT,
        longitude=EVENT_LON,
        maxradius=kilometers2degrees(RADIUS_KM),
        minmagnitude=MIN_CATALOG_MAG,
        orderby="time-asc",
    )
    cat_regional.write(
        str(DIRS["event"] / f"{EVENT_ID}_regional_catalog.xml"),
        format="QUAKEML"
    )

    rows = []
    for ev in cat_regional:
        org = ev.preferred_origin() or (ev.origins[0] if ev.origins else None)
        mag = ev.preferred_magnitude() or (ev.magnitudes[0] if ev.magnitudes else None)
        if org is None:
            continue
        dist_m, az, baz = gps2dist_azimuth(
            EVENT_LAT, EVENT_LON, org.latitude, org.longitude
        )
        rows.append({
            "time": str(org.time),
            "latitude": org.latitude,
            "longitude": org.longitude,
            "depth_km": None if org.depth is None else org.depth / 1000.0,
            "magnitude": None if mag is None else mag.mag,
            "mag_type": None if mag is None else mag.magnitude_type,
            "distance_km_from_mainshock": dist_m / 1000.0,
            "azimuth_deg_from_mainshock": az,
            "resource_id": str(ev.resource_id),
            "after_mainshock": org.time >= EVENT_TIME,
        })

    regional_df = pd.DataFrame(rows)
    regional_df.to_csv(
        DIRS["tables"] / f"{EVENT_ID}_regional_catalog.csv", index=False
    )
    print("Regional events:", len(regional_df))
    display(regional_df.tail(20))
except Exception as e:
    regional_df = pd.DataFrame()
    print("Regional catalog request failed:", repr(e))



## 4. FDSN waveform + StationXML mass download within 500 km

Three passes are used so a station can contribute more than one useful sensor family:

- high gain / broadband: HH, BH, EH
- strong motion: HN, BN, EN
- short period: SH

The downloader tries ObsPy-known FDSN data centers and explicitly adds KOERI's EIDA endpoint. Restricted data still require authorization from the network operator.


In [ ]:

EXCLUDE_PROVIDERS = {"RASPISHAKE", "IRISPH5"}
provider_specs = [
    name for name in sorted(URL_MAPPINGS.keys())
    if name.upper() not in EXCLUDE_PROVIDERS
]

KOERI_FDSN_BASE = "https://eida.koeri.boun.edu.tr"
provider_specs = [KOERI_FDSN_BASE] + provider_specs
provider_specs = list(dict.fromkeys(provider_specs))

print("Providers/endpoints to try:", len(provider_specs))
print(provider_specs[:25])


In [ ]:

domain = CircularDomain(
    latitude=EVENT_LAT,
    longitude=EVENT_LON,
    minradius=0.0,
    maxradius=kilometers2degrees(RADIUS_KM),
)

family_priorities = {
    "high_gain": ["HH[ZNE12]", "BH[ZNE12]", "EH[ZNE12]"],
    "strong_motion": ["HN[ZNE12]", "BN[ZNE12]", "EN[ZNE12]"],
    "short_period": ["SH[ZNE12]"],
}

REJECT_CHANNELS_WITH_GAPS = True
MINIMUM_LENGTH_FRACTION = 0.90

mdl = MassDownloader(providers=provider_specs)

for family, priorities in family_priorities.items():
    print("\n" + "=" * 90)
    print("Downloading family:", family, priorities)

    restrictions = Restrictions(
        starttime=EVENT_TIME - PRE_EVENT_SECONDS,
        endtime=EVENT_TIME + POST_EVENT_SECONDS,
        reject_channels_with_gaps=REJECT_CHANNELS_WITH_GAPS,
        minimum_length=MINIMUM_LENGTH_FRACTION,
        minimum_interstation_distance_in_m=0,
        channel_priorities=priorities,
        location_priorities=["", "00", "10", "01", "20"],
    )

    mseed_dir = DIRS["mseed"] / family
    xml_dir = DIRS["stationxml"] / family
    mseed_dir.mkdir(parents=True, exist_ok=True)
    xml_dir.mkdir(parents=True, exist_ok=True)

    try:
        mdl.download(
            domain,
            restrictions,
            mseed_storage=str(mseed_dir),
            stationxml_storage=str(xml_dir),
            threads_per_client=3,
        )
    except Exception as e:
        print(f"Family {family} failed; continuing:", repr(e))
        traceback.print_exc(limit=1)

print("\nFDSN mass download passes finished.")



## 5. Audit StationXML and MiniSEED


In [ ]:

xml_files = sorted(DIRS["stationxml"].rglob("*.xml"))
mseed_files = sorted(DIRS["mseed"].rglob("*.mseed"))

print("StationXML files:", len(xml_files))
print("MiniSEED files:", len(mseed_files))

inventories = []
channel_rows = []

for xf in tqdm(xml_files, desc="Reading StationXML"):
    try:
        inv = read_inventory(str(xf))
        inventories.append(inv)
        family = xf.parent.name

        for net in inv:
            for sta in net:
                dist_m, az, baz = gps2dist_azimuth(
                    EVENT_LAT, EVENT_LON, sta.latitude, sta.longitude
                )
                for cha in sta:
                    channel_rows.append({
                        "family": family,
                        "network": net.code,
                        "station": sta.code,
                        "location": cha.location_code,
                        "channel": cha.code,
                        "latitude": cha.latitude if cha.latitude is not None else sta.latitude,
                        "longitude": cha.longitude if cha.longitude is not None else sta.longitude,
                        "elevation_m": cha.elevation if cha.elevation is not None else sta.elevation,
                        "sample_rate_hz": cha.sample_rate,
                        "start_date": str(cha.start_date) if cha.start_date else None,
                        "end_date": str(cha.end_date) if cha.end_date else None,
                        "distance_km": dist_m / 1000.0,
                        "azimuth_deg": az,
                        "backazimuth_deg": baz,
                        "stationxml_file": str(xf),
                    })
    except Exception as e:
        print("Could not parse", xf, ":", repr(e))

channels_df = pd.DataFrame(channel_rows)

if not channels_df.empty:
    channels_df = (
        channels_df.drop_duplicates(
            subset=["family", "network", "station", "location", "channel"]
        )
        .sort_values(["distance_km", "network", "station", "channel"])
    )
    channels_df.to_csv(
        DIRS["tables"] / "station_channels_500km.csv", index=False
    )

    stations_df = (
        channels_df.sort_values("distance_km")
        .drop_duplicates(subset=["network", "station"])
        [["network", "station", "latitude", "longitude", "elevation_m",
          "distance_km", "azimuth_deg", "backazimuth_deg"]]
        .reset_index(drop=True)
    )
    stations_df.to_csv(
        DIRS["tables"] / "stations_unique_500km.csv", index=False
    )

    print("Unique channels:", len(channels_df))
    print("Unique stations:", len(stations_df))
    display(stations_df.head(40))
else:
    stations_df = pd.DataFrame()
    print("No StationXML metadata found.")


In [ ]:

wf_rows = []

for mf in tqdm(mseed_files, desc="Auditing MiniSEED"):
    try:
        st = read(str(mf), headonly=True)
        for tr in st:
            wf_rows.append({
                "family": mf.parent.name,
                "file": str(mf),
                "id": tr.id,
                "network": tr.stats.network,
                "station": tr.stats.station,
                "location": tr.stats.location,
                "channel": tr.stats.channel,
                "starttime": str(tr.stats.starttime),
                "endtime": str(tr.stats.endtime),
                "sampling_rate": tr.stats.sampling_rate,
                "npts": tr.stats.npts,
                "duration_s": tr.stats.endtime - tr.stats.starttime,
            })
    except Exception as e:
        wf_rows.append({"family": mf.parent.name, "file": str(mf), "error": repr(e)})

waveform_df = pd.DataFrame(wf_rows)
waveform_df.to_csv(DIRS["tables"] / "waveform_files.csv", index=False)
display(waveform_df.head(30))



## 6. Station map


In [ ]:

def destination_point(lat, lon, distance_km, azimuth_deg):
    R = 6371.0088
    phi1 = np.deg2rad(lat)
    lam1 = np.deg2rad(lon)
    d = distance_km / R
    th = np.deg2rad(azimuth_deg)
    phi2 = np.arcsin(
        np.sin(phi1)*np.cos(d) + np.cos(phi1)*np.sin(d)*np.cos(th)
    )
    lam2 = lam1 + np.arctan2(
        np.sin(th)*np.sin(d)*np.cos(phi1),
        np.cos(d) - np.sin(phi1)*np.sin(phi2)
    )
    return np.rad2deg(phi2), (np.rad2deg(lam2) + 540) % 360 - 180

azs = np.linspace(0, 360, 361)
circle = np.array([
    destination_point(EVENT_LAT, EVENT_LON, RADIUS_KM, a) for a in azs
])

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except Exception:
    HAS_CARTOPY = False

fig = plt.figure(figsize=(11, 10))

if HAS_CARTOPY:
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.add_feature(cfeature.BORDERS, linewidth=0.8)
    ax.coastlines(resolution="50m", linewidth=0.8)
    transform = ccrs.PlateCarree()
else:
    ax = plt.axes()
    transform = None

plot_kw = {"transform": transform} if transform is not None else {}

if not stations_df.empty:
    ax.scatter(
        stations_df["longitude"], stations_df["latitude"],
        s=25, marker="^", alpha=0.8,
        label=f"FDSN stations ({len(stations_df)})", **plot_kw
    )

ax.scatter(EVENT_LON, EVENT_LAT, s=180, marker="*", label="M5.3 epicenter", **plot_kw)
ax.plot(circle[:, 1], circle[:, 0], linewidth=1.5, label="500 km radius", **plot_kw)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"{EVENT_ID} — FDSN stations within 500 km")
ax.legend(loc="best")
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(DIRS["figures"] / "stations_500km.png", dpi=220)
plt.show()



## 7. Convert MiniSEED to SAC with event and station headers


In [ ]:

def find_coordinates(trace):
    # Return coordinates and matching inventory for a trace.
    last_error = None
    for inv in inventories:
        try:
            return inv.get_coordinates(trace.id, datetime=trace.stats.starttime), inv
        except Exception as e:
            last_error = e
    raise LookupError(
        f"No StationXML coordinates found for {trace.id}: {last_error}"
    )

sac_written = 0
sac_failed = []

for mf in tqdm(mseed_files, desc="MiniSEED -> SAC"):
    try:
        st = read(str(mf))
    except Exception as e:
        sac_failed.append((str(mf), repr(e)))
        continue

    for tr in st:
        try:
            coords, inv_used = find_coordinates(tr)
            stla = float(coords["latitude"])
            stlo = float(coords["longitude"])
            stel = float(coords.get("elevation", 0.0))

            dist_m, az, baz = gps2dist_azimuth(
                EVENT_LAT, EVENT_LON, stla, stlo
            )
            gcarc = locations2degrees(EVENT_LAT, EVENT_LON, stla, stlo)

            tr.stats.sac = AttribDict({
                "evla": EVENT_LAT,
                "evlo": EVENT_LON,
                "evdp": EVENT_DEPTH_KM,
                "mag": EVENT_MAG,
                "stla": stla,
                "stlo": stlo,
                "stel": stel,
                "dist": dist_m / 1000.0,
                "az": az,
                "baz": baz,
                "gcarc": gcarc,
                "o": float(EVENT_TIME - tr.stats.starttime),
                "kevnm": EVENT_ID[:16],
            })

            family = mf.parent.name
            out_dir = DIRS["sac_raw"] / family
            out_dir.mkdir(parents=True, exist_ok=True)

            loc = tr.stats.location if tr.stats.location else "--"
            start_tag = tr.stats.starttime.strftime("%Y%m%dT%H%M%S")
            fn = (
                f"{tr.stats.network}.{tr.stats.station}.{loc}."
                f"{tr.stats.channel}.{start_tag}.SAC"
            )
            tr.write(str(out_dir / fn), format="SAC")
            sac_written += 1

        except Exception as e:
            sac_failed.append((tr.id, repr(e)))

print("SAC traces written:", sac_written)
print("SAC failures:", len(sac_failed))

if sac_failed:
    display(pd.DataFrame(
        sac_failed, columns=["trace_or_file", "error"]
    ).head(30))



## 8. Optional instrument-response removal to velocity


In [ ]:

MAKE_VELOCITY_SAC = False
PRE_FILT = (0.02, 0.05, 20.0, 25.0)

if MAKE_VELOCITY_SAC:
    vel_written = 0
    failures = []

    for mf in tqdm(mseed_files, desc="Response correction"):
        try:
            st = read(str(mf))
        except Exception as e:
            failures.append((str(mf), repr(e)))
            continue

        for tr in st:
            try:
                coords, inv_used = find_coordinates(tr)
                stla = float(coords["latitude"])
                stlo = float(coords["longitude"])
                dist_m, az, baz = gps2dist_azimuth(
                    EVENT_LAT, EVENT_LON, stla, stlo
                )
                gcarc = locations2degrees(
                    EVENT_LAT, EVENT_LON, stla, stlo
                )

                tr.detrend("demean")
                tr.detrend("linear")
                tr.taper(max_percentage=0.05, type="cosine")
                tr.remove_response(
                    inventory=inv_used,
                    output="VEL",
                    pre_filt=PRE_FILT,
                    water_level=60,
                    zero_mean=False,
                    taper=False,
                )

                tr.stats.sac = AttribDict({
                    "evla": EVENT_LAT,
                    "evlo": EVENT_LON,
                    "evdp": EVENT_DEPTH_KM,
                    "mag": EVENT_MAG,
                    "stla": stla,
                    "stlo": stlo,
                    "stel": float(coords.get("elevation", 0.0)),
                    "dist": dist_m / 1000.0,
                    "az": az,
                    "baz": baz,
                    "gcarc": gcarc,
                    "o": float(EVENT_TIME - tr.stats.starttime),
                    "kevnm": EVENT_ID[:16],
                    "kuser0": "VEL_M_S",
                })

                family = mf.parent.name
                out_dir = DIRS["sac_vel"] / family
                out_dir.mkdir(parents=True, exist_ok=True)
                loc = tr.stats.location if tr.stats.location else "--"
                start_tag = tr.stats.starttime.strftime("%Y%m%dT%H%M%S")
                fn = (
                    f"{tr.stats.network}.{tr.stats.station}.{loc}."
                    f"{tr.stats.channel}.{start_tag}.VEL.SAC"
                )
                tr.write(str(out_dir / fn), format="SAC")
                vel_written += 1

            except Exception as e:
                failures.append((tr.id, repr(e)))

    print("Velocity SAC traces:", vel_written)
    print("Failures:", len(failures))
else:
    print("Skipped. Set MAKE_VELOCITY_SAC = True to run.")



## 9. Quick vertical-component record section


In [ ]:

MAX_TRACES_TO_PLOT = 50
vertical = []

for mf in mseed_files:
    try:
        st = read(str(mf))
        for tr in st:
            if not tr.stats.channel.endswith("Z"):
                continue
            try:
                coords, _ = find_coordinates(tr)
                dist_m, az, baz = gps2dist_azimuth(
                    EVENT_LAT, EVENT_LON,
                    coords["latitude"], coords["longitude"]
                )
                tr2 = tr.copy()
                tr2.detrend("demean")
                tr2.taper(max_percentage=0.03)
                nyq = tr2.stats.sampling_rate / 2.0
                fmax = min(10.0, nyq * 0.8)
                if fmax > 0.2:
                    tr2.filter(
                        "bandpass",
                        freqmin=0.2,
                        freqmax=fmax,
                        corners=3,
                        zerophase=True
                    )
                vertical.append((dist_m / 1000.0, tr2))
            except Exception:
                pass
    except Exception:
        pass

vertical.sort(key=lambda x: x[0])
vertical = vertical[:MAX_TRACES_TO_PLOT]

if vertical:
    fig, ax = plt.subplots(figsize=(12, 10))
    for dist_km, tr in vertical:
        x = tr.times(reftime=EVENT_TIME)
        y = tr.data.astype(float)
        scale = np.nanmax(np.abs(y))
        if not np.isfinite(scale) or scale == 0:
            continue
        y = y / scale
        ax.plot(x, dist_km + y * 3.0, linewidth=0.7)

    ax.axvline(0, linestyle="--", linewidth=1)
    ax.set_xlim(-PRE_EVENT_SECONDS, POST_EVENT_SECONDS)
    ax.set_xlabel("Time from origin (s)")
    ax.set_ylabel("Epicentral distance (km)")
    ax.set_title(f"{EVENT_ID} — normalized vertical record section")
    ax.grid(True, alpha=0.2)
    fig.tight_layout()
    fig.savefig(
        DIRS["figures"] / "record_section_vertical.png", dpi=220
    )
    plt.show()
else:
    print("No vertical traces available.")



# Part B — Coulomb stress scenario

## 10. Extract nodal planes from the current USGS products

The code first checks `moment-tensor`, then `focal-mechanism`. If no strike–dip–rake values are available, it stops instead of inventing a mechanism.


In [ ]:

def product_properties(detail_json, product_name):
    products = (
        detail_json.get("properties", {})
        .get("products", {})
        .get(product_name, [])
    )
    if not products:
        return None
    return products[0].get("properties", {})

def extract_nodal_planes_from_props(props):
    if not props:
        return []

    norm = {
        str(k).lower().replace("_", "-"): v
        for k, v in props.items()
    }

    planes = []
    for i in (1, 2):
        sk = f"nodal-plane-{i}-strike"
        dk = f"nodal-plane-{i}-dip"
        rk = f"nodal-plane-{i}-rake"

        if sk in norm and dk in norm and rk in norm:
            planes.append({
                "name": f"NP{i}",
                "strike": float(norm[sk]),
                "dip": float(norm[dk]),
                "rake": float(norm[rk]),
            })
    return planes

nodal_planes = []
mechanism_source = None

if detail is not None:
    for product_name in ("moment-tensor", "focal-mechanism"):
        pp = product_properties(detail, product_name)
        planes = extract_nodal_planes_from_props(pp)
        if planes:
            nodal_planes = planes
            mechanism_source = product_name
            break

# Manual override: fill only with a published or otherwise justified mechanism.
MANUAL_NODAL_PLANES = [
    # {"name": "manual_NP1", "strike": 0.0, "dip": 90.0, "rake": 0.0},
]

if not nodal_planes and MANUAL_NODAL_PLANES:
    nodal_planes = MANUAL_NODAL_PLANES
    mechanism_source = "manual"

if nodal_planes:
    print("Mechanism source:", mechanism_source)
    display(pd.DataFrame(nodal_planes))
else:
    print("No nodal planes currently available in the retrieved USGS products.")
    print("Re-run later or populate MANUAL_NODAL_PLANES with a published solution.")



## 11. Source scaling and Coulomb parameters

For an event without a published finite-fault model, this notebook estimates a compact rectangular source from seismic moment and an assumed stress drop.

Editable defaults:

- shear modulus = 32 GPa
- Poisson ratio = 0.25
- stress drop = 3 MPa
- source aspect ratio = 2:1
- effective friction = 0.4
- pore pressure change = 0 Pa
- receiver depth = 10 km
- receiver orientation = same as the selected nodal plane

These are model assumptions, not directly measured properties of this earthquake.


In [ ]:

SHEAR_MODULUS_PA = 32e9
POISSON = 0.25
STRESS_DROP_MPA = 3.0
ASPECT_RATIO = 2.0

FRICTION = 0.4
PORE_PRESSURE_CHANGE_PA = 0.0
RECEIVER_DEPTH_KM = 10.0

CFS_HALF_WIDTH_KM = 120.0
GRID_N = 181

def mw_to_m0(mw):
    return 10.0 ** (1.5 * mw + 9.1)

def estimate_rectangular_source(
    mw, mu_pa, stress_drop_mpa=3.0, aspect_ratio=2.0
):
    # Circular-crack stress-drop scaling converted to a rectangle.
    m0 = mw_to_m0(mw)
    ds = stress_drop_mpa * 1e6
    r = (7.0 * m0 / (16.0 * ds)) ** (1.0 / 3.0)
    area = np.pi * r**2
    length = np.sqrt(area * aspect_ratio)
    width = area / length
    slip = m0 / (mu_pa * area)

    return {
        "M0_Nm": m0,
        "equivalent_crack_radius_km": r / 1000.0,
        "area_km2": area / 1e6,
        "length_km": length / 1000.0,
        "width_km": width / 1000.0,
        "mean_slip_m": slip,
    }

source_scale = estimate_rectangular_source(
    EVENT_MAG, SHEAR_MODULUS_PA, STRESS_DROP_MPA, ASPECT_RATIO
)
display(pd.DataFrame([source_scale]))



## 12. Okada stress tensor and ΔCFS


In [ ]:

from pyrocko.modelling import OkadaSource, okada_ext

def lame_lambda(mu, nu):
    return 2.0 * mu * nu / (1.0 - 2.0 * nu)

def receiver_vectors(strike, dip, rake):
    d2r = np.pi / 180.0
    strike_rad = strike * d2r
    dip_rad = dip * d2r
    rake_rad = rake * d2r

    ns = np.zeros(3)
    rst = np.zeros(3)
    rdi = np.zeros(3)

    ns[0] = np.sin(dip_rad) * np.cos(strike_rad + 0.5*np.pi)
    ns[1] = np.sin(dip_rad) * np.sin(strike_rad + 0.5*np.pi)
    ns[2] = -np.cos(dip_rad)

    rst[0] = np.cos(strike_rad)
    rst[1] = np.sin(strike_rad)
    rst[2] = 0.0

    rdi[0] = np.cos(dip_rad) * np.cos(strike_rad + 0.5*np.pi)
    rdi[1] = np.cos(dip_rad) * np.sin(strike_rad + 0.5*np.pi)
    rdi[2] = np.sin(dip_rad)

    ts = rst * np.cos(rake_rad) - rdi * np.sin(rake_rad)
    return ns, ts

def okada_cfs_grid(
    source_strike,
    source_dip,
    source_rake,
    receiver_strike=None,
    receiver_dip=None,
    receiver_rake=None,
    half_width_km=CFS_HALF_WIDTH_KM,
    grid_n=GRID_N,
    receiver_depth_km=RECEIVER_DEPTH_KM,
    friction=FRICTION,
    pressure_pa=PORE_PRESSURE_CHANGE_PA,
):
    if receiver_strike is None:
        receiver_strike = source_strike
        receiver_dip = source_dip
        receiver_rake = source_rake

    L = source_scale["length_km"] * 1000.0
    W = source_scale["width_km"] * 1000.0
    D = source_scale["mean_slip_m"]

    src = OkadaSource(
        lat=EVENT_LAT,
        lon=EVENT_LON,
        north_shift=0.0,
        east_shift=0.0,
        depth=EVENT_DEPTH_KM * 1000.0,
        al1=-L/2.0,
        al2=L/2.0,
        aw1=-W/2.0,
        aw2=W/2.0,
        strike=float(source_strike),
        dip=float(source_dip),
        rake=float(source_rake),
        slip=float(D),
        opening=0.0,
        poisson=POISSON,
        shearmod=SHEAR_MODULUS_PA,
    )

    norths = np.linspace(-half_width_km, half_width_km, grid_n) * 1000.0
    easts = np.linspace(-half_width_km, half_width_km, grid_n) * 1000.0
    EE, NN = np.meshgrid(easts, norths)

    receiver = np.column_stack([
        NN.ravel(),
        EE.ravel(),
        np.full(EE.size, receiver_depth_km * 1000.0),
    ])

    source_patch = src.source_patch()[None, :]
    source_disl = src.source_disloc()[None, :]
    lam = lame_lambda(SHEAR_MODULUS_PA, POISSON)

    result = okada_ext.okada(
        source_patch,
        source_disl,
        receiver,
        lam,
        SHEAR_MODULUS_PA,
        nthreads=0,
        rotate_sdn=False,
        stack_sources=True,
    )

    if result.ndim != 2 or result.shape[1] < 12:
        raise RuntimeError(f"Unexpected Okada result shape: {result.shape}")

    grad = result[:, 3:12]
    grad_T = result[:, (3, 6, 9, 4, 7, 10, 5, 8, 11)]
    eps = 0.5 * (grad + grad_T)

    diag = [0, 4, 8]
    dil = eps[:, diag].sum(axis=1)[:, None]
    kron = np.zeros(9)
    kron[diag] = 1.0
    stress = kron[None, :] * lam * dil + 2.0 * SHEAR_MODULUS_PA * eps

    ns, ts = receiver_vectors(
        receiver_strike, receiver_dip, receiver_rake
    )

    sigma_n = np.sum(
        np.tile(ns, 3) * stress * np.repeat(ns, 3),
        axis=1
    )
    tau = np.sum(
        np.tile(ts, 3) * stress * np.repeat(ns, 3),
        axis=1
    )

    cfs = tau + friction * (sigma_n + pressure_pa)

    return {
        "source": src,
        "easts_m": easts,
        "norths_m": norths,
        "EE_m": EE,
        "NN_m": NN,
        "cfs_pa": cfs.reshape(grid_n, grid_n),
        "sigma_n_pa": sigma_n.reshape(grid_n, grid_n),
        "tau_pa": tau.reshape(grid_n, grid_n),
        "receiver_plane": {
            "strike": receiver_strike,
            "dip": receiver_dip,
            "rake": receiver_rake,
        },
    }



## 13. Compute and save ΔCFS maps for all available nodal-plane scenarios

Positive ΔCFS means increased failure tendency only for the defined receiver orientation and model assumptions. Negative values mean decreased tendency under the same assumptions.


In [ ]:

local_crs = CRS.from_proj4(
    f"+proj=aeqd +lat_0={EVENT_LAT} +lon_0={EVENT_LON} "
    "+datum=WGS84 +units=m +no_defs"
)
wgs84 = CRS.from_epsg(4326)
to_geo = Transformer.from_crs(local_crs, wgs84, always_xy=True)

cfs_results = {}

if not nodal_planes:
    raise RuntimeError(
        "No focal mechanism/nodal planes available. Re-run later or "
        "populate MANUAL_NODAL_PLANES with a published strike/dip/rake."
    )

for plane in nodal_planes:
    print("\nCalculating", plane)

    res = okada_cfs_grid(
        source_strike=plane["strike"],
        source_dip=plane["dip"],
        source_rake=plane["rake"],
    )
    cfs_results[plane["name"]] = res

    EE = res["EE_m"]
    NN = res["NN_m"]
    lon_grid, lat_grid = to_geo.transform(EE, NN)
    cfs_mpa = res["cfs_pa"] / 1e6

    np.savez_compressed(
        DIRS["coulomb"] / f"{EVENT_ID}_{plane['name']}_cfs.npz",
        longitude=lon_grid,
        latitude=lat_grid,
        east_km=EE / 1000.0,
        north_km=NN / 1000.0,
        depth_km=RECEIVER_DEPTH_KM,
        cfs_mpa=cfs_mpa,
        normal_stress_mpa=res["sigma_n_pa"] / 1e6,
        shear_stress_mpa=res["tau_pa"] / 1e6,
        strike=plane["strike"],
        dip=plane["dip"],
        rake=plane["rake"],
        friction=FRICTION,
        pressure_change_pa=PORE_PRESSURE_CHANGE_PA,
        source_length_km=source_scale["length_km"],
        source_width_km=source_scale["width_km"],
        source_mean_slip_m=source_scale["mean_slip_m"],
        assumed_stress_drop_mpa=STRESS_DROP_MPA,
    )

    finite = np.isfinite(cfs_mpa)
    if finite.any():
        vmax = np.nanpercentile(np.abs(cfs_mpa[finite]), 98)
        if not np.isfinite(vmax) or vmax == 0:
            vmax = np.nanmax(np.abs(cfs_mpa[finite]))
    else:
        vmax = 1.0

    fig, ax = plt.subplots(figsize=(10, 9))
    mesh = ax.pcolormesh(
        lon_grid,
        lat_grid,
        cfs_mpa,
        shading="auto",
        cmap="RdBu_r",
        norm=TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax),
    )

    ax.scatter(
        EVENT_LON, EVENT_LAT,
        s=160, marker="*", label="Mainshock"
    )

    strike_rad = np.deg2rad(plane["strike"])
    halfL_m = source_scale["length_km"] * 500.0
    n1, n2 = -halfL_m*np.cos(strike_rad), halfL_m*np.cos(strike_rad)
    e1, e2 = -halfL_m*np.sin(strike_rad), halfL_m*np.sin(strike_rad)
    flon, flat = to_geo.transform([e1, e2], [n1, n2])
    ax.plot(
        flon, flat, linewidth=3,
        label=f"{plane['name']} source strike"
    )

    if "regional_df" in globals() and not regional_df.empty:
        rr = regional_df.copy()
        try:
            rr["time_dt"] = pd.to_datetime(rr["time"], utc=True)
            origin_dt = pd.Timestamp(EVENT_TIME.datetime, tz="UTC")
            aft = rr[
                (rr["time_dt"] > origin_dt) &
                (rr["distance_km_from_mainshock"] <= CFS_HALF_WIDTH_KM)
            ]
            if not aft.empty:
                ax.scatter(
                    aft["longitude"],
                    aft["latitude"],
                    s=np.clip(
                        (aft["magnitude"].fillna(1.0).values**2) * 6,
                        8, 120
                    ),
                    facecolors="none",
                    edgecolors="k",
                    linewidths=0.8,
                    label=f"Aftershocks ({len(aft)})"
                )
        except Exception:
            pass

    cb = fig.colorbar(mesh, ax=ax, shrink=0.82)
    cb.set_label("ΔCFS (MPa)")

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(
        f"{EVENT_ID} — Coulomb scenario {plane['name']}\n"
        f"source/receiver strike={plane['strike']:.1f}°, "
        f"dip={plane['dip']:.1f}°, rake={plane['rake']:.1f}°, "
        f"receiver depth={RECEIVER_DEPTH_KM:.1f} km"
    )
    ax.legend(loc="best")
    ax.grid(True, alpha=0.2)
    fig.tight_layout()
    fig.savefig(
        DIRS["figures"] / f"{EVENT_ID}_{plane['name']}_coulomb.png",
        dpi=250
    )
    plt.show()



## 14. Optional Coulomb sensitivity ensemble

This tests multiple stress drops and effective friction coefficients. A publication-quality interpretation should use sensitivity analysis rather than a single parameter set.


In [ ]:

RUN_CFS_SENSITIVITY = False

STRESS_DROPS_MPA = [1.0, 3.0, 5.0]
FRICTIONS = [0.2, 0.4, 0.6]

if RUN_CFS_SENSITIVITY:
    sensitivity_rows = []

    for plane in nodal_planes:
        for ds in STRESS_DROPS_MPA:
            for mu_eff in FRICTIONS:
                source_scale_backup = source_scale
                source_scale = estimate_rectangular_source(
                    EVENT_MAG,
                    SHEAR_MODULUS_PA,
                    ds,
                    ASPECT_RATIO
                )
                try:
                    res = okada_cfs_grid(
                        source_strike=plane["strike"],
                        source_dip=plane["dip"],
                        source_rake=plane["rake"],
                        friction=mu_eff,
                    )
                    c = res["cfs_pa"] / 1e6
                    sensitivity_rows.append({
                        "plane": plane["name"],
                        "stress_drop_mpa": ds,
                        "friction": mu_eff,
                        "max_cfs_mpa": np.nanmax(c),
                        "min_cfs_mpa": np.nanmin(c),
                        "p95_abs_cfs_mpa": np.nanpercentile(np.abs(c), 95),
                        "source_length_km": source_scale["length_km"],
                        "source_width_km": source_scale["width_km"],
                        "mean_slip_m": source_scale["mean_slip_m"],
                    })
                finally:
                    source_scale = source_scale_backup

    sensitivity_df = pd.DataFrame(sensitivity_rows)
    sensitivity_df.to_csv(
        DIRS["coulomb"] / "cfs_sensitivity_summary.csv",
        index=False
    )
    display(sensitivity_df)
else:
    print("Skipped. Set RUN_CFS_SENSITIVITY = True to run.")



## 15. Dataset inventory


In [ ]:

inventory_rows = []

for p in sorted(ROOT.rglob("*")):
    if p.is_file():
        inventory_rows.append({
            "relative_path": str(p.relative_to(ROOT)),
            "size_bytes": p.stat().st_size,
            "size_mb": p.stat().st_size / (1024**2),
        })

file_inventory = pd.DataFrame(inventory_rows)
file_inventory.to_csv(ROOT / "DATASET_INVENTORY.csv", index=False)

print("Files:", len(file_inventory))
print("Total size (GB):", file_inventory["size_bytes"].sum() / (1024**3))
display(file_inventory.tail(50))



# Publication checklist

Before interpreting the Coulomb map as a scientific result:

1. Confirm the reviewed origin and magnitude after USGS/AFAD/KOERI updates.
2. Use a reviewed moment tensor or focal mechanism.
3. Determine which nodal plane is the actual rupture plane using mapped faults, relocated aftershocks, InSAR/GNSS, or other constraints.
4. Replace scenario dimensions with a finite-source/geodetic model if one becomes available.
5. Test multiple receiver-fault orientations.
6. Test friction, stress drop, source depth, and elastic parameters.
7. Compare ΔCFS with aftershocks only after considering catalog completeness and location uncertainty.
8. Do not treat positive ΔCFS by itself as an earthquake prediction.
